In [1]:
from __future__ import annotations
import os
import re
import io
import json
import csv
import itertools
import mimetypes
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Iterator, Union
from datetime import datetime
from dataclasses import dataclass, field, asdict
import argparse
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading
from collections import Counter
import time
import sys
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# Определение GPU доступности
# ============================================================================

def is_gpu_available():
    """Проверка доступности GPU"""
    try:
        import torch
        if torch.cuda.is_available():
            return True, f"CUDA ({torch.cuda.get_device_name(0)})"
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            return True, "MPS (Apple Silicon)"
    except:
        pass
    
    try:
        import tensorflow as tf
        if tf.config.list_physical_devices('GPU'):
            return True, "TensorFlow GPU"
    except:
        pass
    
    return False, "CPU"

GPU_AVAILABLE, GPU_NAME = is_gpu_available()

# ============================================================================
# Конфигурации и константы
# ============================================================================

INCLUDE_EXTS = {'doc', 'docx', 'gif', 'html', 'ipynb', 'jpeg', 'jpg', 'pdf', 
                'php', 'png', 'rtf', 'xls', 'xlsx', 'txt', 'csv', 'json', 'tif', 'tiff'}

PDN_CATEGORIES = ['обычные', 'государственные', 'платёжные', 'биометрические', 'специальные']

# ============================================================================
# Spark инициализация с GPU
# ============================================================================

SPARK_AVAILABLE = False
try:
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import udf, col, size, when, lit
    from pyspark.sql.types import (
        StructType, StructField, StringType, IntegerType, 
        MapType, BooleanType, LongType, ArrayType
    )
    SPARK_AVAILABLE = True
except ImportError:
    pass

def get_spark_session(app_name="PDNScanner", use_gpu=True):
    """Создание Spark сессии с поддержкой GPU"""
    if not SPARK_AVAILABLE:
        return None
    
    builder = SparkSession.builder.appName(app_name)
    
    if use_gpu and GPU_AVAILABLE:
        builder = builder \
            .config("spark.executor.resource.gpu.amount", "1") \
            .config("spark.task.resource.gpu.amount", "1") \
            .config("spark.executor.cores", "2") \
            .config("spark.sql.adaptive.enabled", "true") \
            .config("spark.driver.memory", "4g") \
            .config("spark.executor.memory", "4g")
        
        if "CUDA" in GPU_NAME:
            try:
                builder = builder.config("spark.plugins", "com.nvidia.spark.SQLPlugin")
            except:
                pass
    
    builder = builder \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.driver.memory", "4g") \
        .config("spark.executor.memory", "4g")
    
    try:
        spark = builder.getOrCreate()
        spark.sparkContext.setLogLevel("ERROR")
        return spark
    except Exception as e:
        print(f"Ошибка создания Spark сессии: {e}")
        return None

# ============================================================================
# YOLO инициализация с GPU
# ============================================================================

YOLO_AVAILABLE = False
_yolo_model = None
_yolo_lock = threading.Lock()
_yolo_initialized = False

try:
    from ultralytics import YOLO
    import cv2
    import numpy as np
    from PIL import Image
    YOLO_AVAILABLE = True
    
    def get_yolo_model():
        """Потокобезопасная загрузка YOLO модели (один раз)"""
        global _yolo_model, _yolo_initialized
        
        if _yolo_initialized:
            return _yolo_model
        
        with _yolo_lock:
            if _yolo_initialized:
                return _yolo_model
            
            try:
                device = 'cuda' if GPU_AVAILABLE and 'CUDA' in GPU_NAME else ('mps' if GPU_AVAILABLE and 'MPS' in GPU_NAME else 'cpu')
                print(f"Загрузка YOLO модели на устройство: {device}")
                
                model = None
                try:
                    model_path = 'yolov8n.pt'
                    if not os.path.exists(model_path):
                        print(f"Скачивание модели {model_path}...")
                    model = YOLO(model_path)
                    if model:
                        print(f"✅ Загружена модель: {model_path}")
                except Exception as e:
                    print(f"Не удалось загрузить модель: {e}")
                    return None
                
                if model is None:
                    print("⚠️ Не удалось загрузить ни одну модель YOLO")
                    return None
                
                _yolo_model = model
                _yolo_initialized = True
                return _yolo_model
                
            except Exception as e:
                print(f"⚠️ Критическая ошибка загрузки YOLO: {e}")
                return None
    
except ImportError as e:
    print(f"Ultralytics YOLO не установлен: {e}")
    def get_yolo_model():
        return None

# ============================================================================
# Вспомогательные функции
# ============================================================================

def detect_encoding(raw_bytes: bytes) -> str:
    try:
        import chardet
        res = chardet.detect(raw_bytes)
        return res.get('encoding') or 'utf-8'
    except:
        return 'utf-8'

def luhn_check(number: str) -> bool:
    digits = [int(d) for d in re.sub(r'\D', '', number)]
    if not (13 <= len(digits) <= 19):
        return False
    s = 0
    parity = len(digits) % 2
    for i, d in enumerate(digits):
        if i % 2 == parity:
            d *= 2
            if d > 9:
                d -= 9
        s += d
    return s % 10 == 0

def snils_valid(snils: str) -> bool:
    nums = re.sub(r'\D', '', snils)
    if len(nums) != 11:
        return False
    base = [int(x) for x in nums[:9]]
    check = int(nums[9:])
    s = sum((9 - i) * d for i, d in enumerate(base))
    if s < 100:
        c = s
    elif s in (100, 101):
        c = 0
    else:
        c = s % 101
        if c == 100:
            c = 0
    return c == check

def inn_valid(inn: str) -> bool:
    nums = re.sub(r'\D', '', inn)
    if len(nums) == 10:
        w = [2, 4, 10, 3, 5, 9, 4, 6, 8]
        c = sum(int(nums[i]) * w[i] for i in range(9)) % 11 % 10
        return c == int(nums[9])
    elif len(nums) == 12:
        w1 = [7, 2, 4, 10, 3, 5, 9, 4, 6, 8, 0]
        w2 = [3, 7, 2, 4, 10, 3, 5, 9, 4, 6, 8, 0]
        c1 = sum(int(nums[i]) * w1[i] for i in range(11)) % 11 % 10
        c2 = sum(int(nums[i]) * w2[i] for i in range(11)) % 11 % 10
        return c1 == int(nums[10]) and c2 == int(nums[11])
    return False

def has_context(text: str, idx: int, window: int, *keywords: str) -> bool:
    start = max(0, idx - window)
    end = min(len(text), idx + window)
    chunk = text[start:end].lower()
    return any(k.lower() in chunk for k in keywords)

# ============================================================================
# Детекция лиц с помощью YOLO на GPU
# ============================================================================

def detect_faces_in_image_with_model(image_bytes: bytes, model) -> Tuple[int, List[Dict]]:
    """Детекция лиц с использованием переданной модели (потокобезопасно)"""
    if model is None:
        return 0, []
    
    try:
        nparr = np.frombuffer(image_bytes, np.uint8)
        img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
        
        if img is None:
            return 0, []
        
        height, width = img.shape[:2]
        if width > 640 or height > 640:
            scale = min(640/width, 640/height)
            new_width = int(width * scale)
            new_height = int(height * scale)
            img = cv2.resize(img, (new_width, new_height))
        
        results = model(img, conf=0.25, verbose=False)
        
        face_count = 0
        for result in results:
            boxes = result.boxes
            if boxes is not None:
                face_count = len(boxes)
        
        return face_count, []
        
    except Exception:
        return 0, []

# ============================================================================
# Извлечение текста из файлов
# ============================================================================

def extract_text_from_bytes(content: bytes, ext: str, detect_faces: bool = True, yolo_model=None) -> Tuple[str, int]:
    face_count = 0
    
    try:
        if ext in {'txt', 'csv', 'json', 'log', 'md', 'py', 'js', 'html', 'xml', 'php', 'rtf'}:
            for enc in ['utf-8', 'cp1251', 'latin-1']:
                try:
                    return content.decode(enc, errors='ignore'), 0
                except:
                    continue
            return content.decode('utf-8', errors='ignore'), 0
        
        elif ext == 'pdf':
            text = ''
            try:
                import PyPDF2
                import io
                reader = PyPDF2.PdfReader(io.BytesIO(content))
                for page in reader.pages:
                    try:
                        page_text = page.extract_text()
                        if page_text:
                            text += page_text + '\n'
                    except:
                        pass
                if text.strip():
                    return text, 0
            except:
                pass
            
            try:
                from pdfminer.high_level import extract_text as pdfminer_extract
                import io
                text = pdfminer_extract(io.BytesIO(content)) or ''
                if text.strip():
                    return text, 0
            except:
                pass
            
            return '', 0
        
        elif ext in {'xls', 'xlsx', 'xlsm'}:
            try:
                import pandas as pd
                import io
                df = pd.read_excel(io.BytesIO(content), header=None, dtype=str)
                return ' '.join(df.stack().astype(str).tolist()), 0
            except:
                return '', 0
        
        elif ext == 'docx':
            try:
                import docx
                import io
                doc = docx.Document(io.BytesIO(content))
                parts = []
                for p in doc.paragraphs:
                    if p.text:
                        parts.append(p.text)
                for tbl in doc.tables:
                    for row in tbl.rows:
                        row_text = ' \t '.join(cell.text for cell in row.cells if cell.text)
                        if row_text:
                            parts.append(row_text)
                return '\n'.join(parts), 0
            except:
                return '', 0
        
        elif ext == 'doc':
            try:
                text = content.decode('latin-1', errors='ignore')
                text = re.sub(r'[^\x20-\x7E\u0400-\u04FF]+', ' ', text)
                return re.sub(r'\s+', ' ', text), 0
            except:
                return '', 0
        
        elif ext == 'html':
            try:
                txt = content.decode('utf-8', errors='ignore')
                from bs4 import BeautifulSoup
                soup = BeautifulSoup(txt, 'html.parser')
                return soup.get_text(' '), 0
            except:
                return content.decode('utf-8', errors='ignore'), 0
        
        elif ext == 'ipynb':
            try:
                data = json.loads(content.decode('utf-8', errors='ignore'))
                parts = []
                for cell in data.get('cells', []):
                    src = cell.get('source', [])
                    if isinstance(src, list):
                        parts.append(''.join(src))
                    elif isinstance(src, str):
                        parts.append(src)
                return '\n'.join(parts), 0
            except:
                return '', 0
        
        elif ext in {'jpg', 'jpeg', 'png', 'gif', 'tif', 'tiff', 'bmp'}:
            text = ''
            try:
                import pytesseract
                from PIL import Image
                import io
                img = Image.open(io.BytesIO(content))
                if img.mode not in ('RGB', 'L'):
                    img = img.convert('RGB')
                if img.width * img.height > 4000000:
                    img.thumbnail((2000, 2000))
                text = pytesseract.image_to_string(img, lang='rus+eng')
            except:
                pass
            
            if detect_faces and yolo_model:
                face_count, _ = detect_faces_in_image_with_model(content, yolo_model)
            
            return text, face_count
        
        else:
            return content.decode('utf-8', errors='ignore'), 0
            
    except Exception as e:
        return '', 0

# ============================================================================
# Детекторы ПДн
# ============================================================================

EMAIL_RE = re.compile(r"\b[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[A-Za-z]{2,}\b")
PHONE_RE = re.compile(r"(?:(?:\+7|8)\s*\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{2}[\s-]?\d{2})")
FIO_RE = re.compile(r"\b[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+(?:\s+[А-ЯЁ][а-яё]+)?\b")
DOB_RE = re.compile(r"\b(\d{2}[./]\d{2}[./]\d{4})\b")
INDEX_RE = re.compile(r"\b\d{6}\b")
SNILS_RE = re.compile(r"\b\d{3}-\d{3}-\d{3}\s?\d{2}\b|\b\d{11}\b")
INN10_RE = re.compile(r"(?<!\d)\d{10}(?!\d)")
INN12_RE = re.compile(r"(?<!\d)\d{12}(?!\d)")
PASSPORT_RE = re.compile(r"(?:(?<!\d)\d{2}\s?\d{2}\s?\d{6}(?!\d))")
CARD_RE = re.compile(r"(?:(?:\d[ -]*?){13,19})")
RS_RE = re.compile(r"(?i)(?:р/с|расч[её]тн(?:ый)?\s+сч[её]т)[^\d]*(\d{20})")
BIK_RE = re.compile(r"(?i)бик[^\d]*(\d{9})")
CVV_RE = re.compile(r"\b(CVV|CVC|CVV2)\b", re.IGNORECASE)

BIOMETRIC_KEYS = [
    'биометр', 'отпечат', 'радуж', 'ирис', 'лицев', 'селфи', 'faceid', 
    'fingerprint', 'iris', 'voiceprint', 'голосов', 'геометрия лица'
]

SPECIAL_KEYS = [
    'диагноз', 'анамнез', 'инвалид', 'здоровь', 'медицин', 'психиатр', 
    'вич', 'религ', 'вероисповед', 'политическ', 'партия', 'интим', 'сексуаль'
]

def count_occurrences(pattern: re.Pattern, text: str) -> int:
    return len(list(pattern.finditer(text)))

def detect_categories(text: str, face_count: int = 0) -> Dict[str, int]:
    t = text if isinstance(text, str) else ''
    low = t.lower()
    cats = {
        'обычные': 0,
        'государственные': 0,
        'платёжные': 0,
        'биометрические': 0,
        'специальные': 0
    }
    
    if face_count > 0:
        cats['биометрические'] += face_count
    
    if not t:
        return cats
    
    cats['обычные'] += count_occurrences(EMAIL_RE, t)
    cats['обычные'] += count_occurrences(PHONE_RE, t)
    cats['обычные'] += min(5, count_occurrences(FIO_RE, t))
    
    for m in DOB_RE.finditer(t):
        if has_context(low, m.start(), 40, 'дата рождения', 'родил', 'д.р.', 'birth'):
            cats['обычные'] += 1
    
    for m in INDEX_RE.finditer(t):
        if has_context(low, m.start(), 40, 'ул', 'улица', 'просп', 'пер', 'дом', 'квартира', 'город', 'г.', 'адрес'):
            cats['обычные'] += 1
    
    for m in SNILS_RE.finditer(t):
        if snils_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in INN10_RE.finditer(t):
        if inn_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in INN12_RE.finditer(t):
        if inn_valid(m.group(0)):
            cats['государственные'] += 1
    
    for m in PASSPORT_RE.finditer(t):
        if has_context(low, m.start(), 50, 'паспорт', 'серия', 'номер', 'код подразделения'):
            cats['государственные'] += 1
    
    for m in CARD_RE.finditer(t):
        raw = m.group(0)
        digits = re.sub(r'\D', '', raw)
        if 13 <= len(digits) <= 19 and luhn_check(raw):
            if has_context(low, m.start(), 40, 'visa', 'mastercard', 'карта', 'cvv', 'cvc', 'номер карты'):
                cats['платёжные'] += 1
    
    for m in RS_RE.finditer(t):
        cats['платёжные'] += 1
    
    for m in BIK_RE.finditer(t):
        cats['платёжные'] += 1
    
    if CVV_RE.search(t):
        cats['платёжные'] += 1
    
    if any(k in low for k in BIOMETRIC_KEYS):
        cats['биометрические'] += 1
    
    if any(k in low for k in SPECIAL_KEYS):
        cats['специальные'] += 1
    
    return cats

def estimate_uz(cats: Dict[str, int]) -> str:
    total = sum(cats.values())
    distinct = sum(1 for v in cats.values() if v > 0)
    has_special = cats['специальные'] > 0
    has_bio = cats['биометрические'] > 0
    has_pay = cats['платёжные'] > 0
    has_gov = cats['государственные'] > 0
    has_common = cats['обычные'] > 0
    
    if has_special or has_bio:
        return 'УЗ-1' if (total >= 5 or distinct >= 2) else 'УЗ-2'
    if has_pay or has_gov:
        return 'УЗ-2' if (total >= 5 or distinct >= 2) else 'УЗ-3'
    if has_common:
        return 'УЗ-3' if (total >= 5 or distinct >= 2) else 'УЗ-4'
    return 'нет ПДн'

# ============================================================================
# Анализ файла
# ============================================================================

def analyze_file_content(file_path: str, content: bytes, detect_faces: bool = True, yolo_model=None) -> Dict[str, Any]:
    path = Path(file_path)
    
    result = {
        'путь': str(file_path),
        'категории_ПДн': '',
        'количество_находок': 0,
        'УЗ': 'нет ПДн',
        'формат_файла': '',
        'success': False,
        'error': None
    }
    
    if content is None:
        result['error'] = 'Пустое содержимое'
        return result
    
    ext = path.suffix.lower().lstrip('.')
    result['формат_файла'] = ext
    
    if ext not in INCLUDE_EXTS:
        result['error'] = f'Неподдерживаемый формат: {ext}'
        return result
    
    try:
        text, face_count = extract_text_from_bytes(content, ext, detect_faces, yolo_model)
        
        cats = detect_categories(text, face_count)
        
        # Формируем строку категорий
        categories_str = ', '.join([f"{k}:{v}" for k, v in cats.items() if v > 0])
        result['категории_ПДн'] = categories_str if categories_str else 'нет'
        result['количество_находок'] = sum(cats.values())
        result['УЗ'] = estimate_uz(cats)
        result['success'] = True
        
    except Exception as e:
        result['error'] = str(e)[:200]
    
    return result

# ============================================================================
# Класс сканера с поддержкой Spark и GPU
# ============================================================================

class PDNScanner:
    def __init__(self, max_workers: int = 60, use_spark: bool = True, detect_faces: bool = True):
        self.max_workers = max_workers
        self.use_spark = use_spark and SPARK_AVAILABLE
        self.detect_faces = detect_faces and YOLO_AVAILABLE
        self.results = []
        self.spark = None
        self.yolo_model = None
        
        if self.detect_faces:
            print("Инициализация YOLO для детекции лиц...")
            self.yolo_model = get_yolo_model()
            if self.yolo_model:
                print(f"✅ YOLO модель загружена (GPU: {GPU_AVAILABLE})")
            else:
                print("⚠️ Не удалось загрузить YOLO модель, детекция лиц отключена")
                self.detect_faces = False
        
        if self.use_spark:
            self.spark = get_spark_session("PDNScanner", use_gpu=GPU_AVAILABLE)
            if self.spark:
                print(f"Spark инициализирован: {self.spark.version}")
                if GPU_AVAILABLE:
                    print(f"Spark GPU поддержка: ВКЛ ({GPU_NAME})")
            else:
                self.use_spark = False
    
    def scan_directory(self, directory_path: str, recursive: bool = True, sample_size: Optional[int] = None) -> pd.DataFrame:
        dir_path = Path(directory_path)
        if not dir_path.exists():
            raise FileNotFoundError(f"Директория не найдена: {directory_path}")
        
        print(f"Сбор файлов из: {dir_path.absolute()}")
        
        files = []
        if recursive:
            for ext in INCLUDE_EXTS:
                pattern = f"*.{ext}"
                files.extend(dir_path.rglob(pattern))
        else:
            for ext in INCLUDE_EXTS:
                pattern = f"*.{ext}"
                files.extend(dir_path.glob(pattern))
        
        files = list(set(files))
        file_paths = [str(f.absolute()) for f in files]
        
        print(f"Найдено файлов: {len(file_paths)}")
        
        if sample_size and sample_size < len(file_paths):
            file_paths = file_paths[:sample_size]
            print(f"Ограничено до: {len(file_paths)} файлов")
        
        if not file_paths:
            print("Нет файлов для анализа")
            return pd.DataFrame()
        
        if self.use_spark and self.spark:
            return self._scan_with_spark(file_paths)
        else:
            return self._scan_with_threads(file_paths)
    
    def _scan_with_spark(self, file_paths: List[str]) -> pd.DataFrame:
        print(f"Анализ файлов с помощью Spark (GPU: {GPU_AVAILABLE})...")
        
        rdd = self.spark.sparkContext.parallelize(file_paths, numSlices=self.max_workers * 2)
        
        def process_file_spark(file_path: str) -> Dict:
            try:
                with open(file_path, 'rb') as f:
                    content = f.read()
                result = analyze_file_content(file_path, content, self.detect_faces, self.yolo_model)
                return result
            except Exception as e:
                return {
                    'путь': file_path,
                    'категории_ПДн': '',
                    'количество_находок': 0,
                    'УЗ': 'нет ПДн',
                    'формат_файла': Path(file_path).suffix.lower().lstrip('.'),
                    'success': False,
                    'error': str(e)[:200]
                }
            
        results_rdd = rdd.map(process_file_spark)
        results = results_rdd.collect()
        
        print(f"Анализ завершен. Обработано {len(results)} файлов")
        
        df = pd.DataFrame(results)
        
        if not df.empty:
            print("\nПример результатов:")
            print(df[['путь', 'категории_ПДн', 'количество_находок', 'УЗ', 'формат_файла']].head(5).to_string())
        
        return df
    
    def _scan_with_threads(self, file_paths: List[str]) -> pd.DataFrame:
        print(f"Анализ файлов (потоков: {self.max_workers})...")
        
        results = []
        lock = threading.Lock()
        processed = 0
        
        def process_file(file_path: str):
            nonlocal processed
            try:
                with open(file_path, 'rb') as f:
                    content = f.read()
                result = analyze_file_content(file_path, content, self.detect_faces, self.yolo_model)
                with lock:
                    processed += 1
                    if processed % 50 == 0:
                        print(f"  Обработано {processed}/{len(file_paths)} файлов")
                return result
            except Exception as e:
                with lock:
                    processed += 1
                return {
                    'путь': file_path,
                    'категории_ПДн': '',
                    'количество_находок': 0,
                    'УЗ': 'нет ПДн',
                    'формат_файла': Path(file_path).suffix.lower().lstrip('.'),
                    'success': False,
                    'error': str(e)[:200]
                }
        
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {executor.submit(process_file, fp): fp for fp in file_paths}
            for future in as_completed(futures):
                results.append(future.result())
        
        print(f"Анализ завершен. Обработано {len(results)} файлов")
        
        df = pd.DataFrame(results)
        
        if not df.empty:
            print("\nПример результатов:")
            print(df[['путь', 'категории_ПДн', 'количество_находок', 'УЗ', 'формат_файла']].head(5).to_string())
        
        return df
    
    def save_report_csv(self, df: pd.DataFrame, output_path: str) -> str:
        """Сохранение отчета в CSV (5 колонок)"""
        if df.empty:
            columns = ['путь', 'категории_ПДн', 'количество_находок', 'УЗ', 'формат_файла']
            pd.DataFrame(columns=columns).to_csv(output_path, index=False, encoding='utf-8-sig')
            return output_path
        
        # Выбираем только нужные колонки
        columns = ['путь', 'категории_ПДн', 'количество_находок', 'УЗ', 'формат_файла']
        report_df = df[columns].copy()
        
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        report_df.to_csv(output_path, index=False, encoding='utf-8-sig')
        print(f"CSV отчет сохранен: {output_path}")
        return output_path
    
    def save_report_json(self, df: pd.DataFrame, output_path: str) -> str:
        """Сохранение отчета в JSON (5 колонок)"""
        if df.empty:
            json_data = {
                'report_generated': datetime.now().isoformat(),
                'total_files': 0,
                'files_with_pdn': 0,
                'results': []
            }
        else:
            columns = ['путь', 'категории_ПДн', 'количество_находок', 'УЗ', 'формат_файла']
            report_df = df[columns].copy()
            
            json_data = {
                'report_generated': datetime.now().isoformat(),
                'gpu_used': GPU_AVAILABLE,
                'gpu_name': GPU_NAME if GPU_AVAILABLE else None,
                'total_files': len(report_df),
                'files_with_pdn': len(report_df[report_df['количество_находок'] > 0]),
                'results': report_df.to_dict(orient='records')
            }
            
            stats = self.get_statistics(df)
            json_data['statistics'] = stats
        
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)
        
        print(f"JSON отчет сохранен: {output_path}")
        return output_path
    
    def save_report_markdown(self, df: pd.DataFrame, output_path: str) -> str:
        """Сохранение отчета в Markdown (5 колонок)"""
        stats = self.get_statistics(df)
        
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        
        md_lines = []
        
        md_lines.append("# Отчет по сканированию персональных данных\n")
        md_lines.append(f"**Дата генерации:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        if GPU_AVAILABLE:
            md_lines.append(f"**GPU:** {GPU_NAME}\n")
        md_lines.append("---\n")
        
        md_lines.append("## Общая статистика\n")
        md_lines.append("| Показатель | Значение |")
        md_lines.append("|------------|----------|")
        md_lines.append(f"| Всего файлов | {stats['total_files']} |")
        md_lines.append(f"| Успешно обработано | {stats['successful']} |")
        md_lines.append(f"| Ошибок | {stats['failed']} |")
        md_lines.append(f"| **Файлов с ПДн** | **{stats['files_with_pdn']}** |")
        md_lines.append(f"| **Всего находок ПДн** | **{stats['total_pdn_hits']}** |")
        md_lines.append(f"| **Обнаружено лиц** | **{stats['total_faces']}** |")
        md_lines.append("")
        
        md_lines.append("## Таблица файлов\n")
        md_lines.append("| путь | категории_ПДн | количество_находок | УЗ | формат_файла |")
        md_lines.append("|------|---------------|-------------------|-----|--------------|")
        
        if not df.empty:
            for _, row in df.head(50).iterrows():
                path_short = Path(row['путь']).name
                categories = row['категории_ПДн'][:50] + '...' if len(row['категории_ПДн']) > 50 else row['категории_ПДн']
                md_lines.append(f"| {path_short} | {categories} | {row['количество_находок']} | {row['УЗ']} | {row['формат_файла']} |")
        
        md_lines.append("")
        
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write('\n'.join(md_lines))
        
        print(f"Markdown отчет сохранен: {output_path}")
        return output_path
    
    def save_all_reports(self, df: pd.DataFrame, base_output_path: str) -> Dict[str, str]:
        """Сохранение отчетов во всех форматах"""
        base_path = Path(base_output_path)
        base_name = base_path.stem
        
        reports = {}
        
        csv_path = base_path.parent / f"{base_name}.csv"
        reports['csv'] = self.save_report_csv(df, str(csv_path))
        
        json_path = base_path.parent / f"{base_name}.json"
        reports['json'] = self.save_report_json(df, str(json_path))
        
        md_path = base_path.parent / f"{base_name}.md"
        reports['markdown'] = self.save_report_markdown(df, str(md_path))
        
        return reports
    
    def get_statistics(self, df: pd.DataFrame) -> Dict[str, Any]:
        if df.empty:
            return {
                'total_files': 0,
                'successful': 0,
                'failed': 0,
                'files_with_pdn': 0,
                'protection_levels': {},
                'category_totals': {cat: 0 for cat in PDN_CATEGORIES},
                'total_pdn_hits': 0,
                'total_faces': 0
            }
        
        total = len(df)
        success = df['success'].sum() if 'success' in df else len(df)
        with_pdn = (df['количество_находок'] > 0).sum()
        total_faces = 0
        
        uz_stats = df['УЗ'].value_counts().to_dict()
        
        return {
            'total_files': total,
            'successful': int(success),
            'failed': total - int(success),
            'files_with_pdn': int(with_pdn),
            'protection_levels': uz_stats,
            'category_totals': {},
            'total_pdn_hits': int(df['количество_находок'].sum()),
            'total_faces': total_faces
        }
    
    def __del__(self):
        if hasattr(self, 'spark') and self.spark:
            try:
                self.spark.stop()
            except:
                pass

# ============================================================================
# Точка входа
# ============================================================================

if __name__ == "__main__":
    # Проверяем, запущен ли скрипт из командной строки или из Jupyter
    is_jupyter = 'ipykernel' in sys.argv[0] if len(sys.argv) > 0 else False
    
    if is_jupyter or len(sys.argv) <= 1:
        # Интерактивный режим для Jupyter
        print("="*60)
        print("СКАНЕР ПЕРСОНАЛЬНЫХ ДАННЫХ")
        print("="*60)
        print(f"GPU доступен: {GPU_AVAILABLE}")
        if GPU_AVAILABLE:
            print(f"GPU: {GPU_NAME}")
        print("="*60)
        
        # Путь к данным
        data_dir = Path.cwd() / "data"
        
        # Создаем тестовую папку если её нет
        if not data_dir.exists():
            data_dir.mkdir()
            test_file = data_dir / "test.txt"
            with open(test_file, 'w', encoding='utf-8') as f:
                f.write("Тестовые данные: Иванов Иван, +7 123 456 78 90, ivanov@example.com")
            print(f"Создан тестовый файл: {test_file}\n")
        
        # Запускаем сканирование
        try:
            scanner = PDNScanner(
                max_workers=100,
                use_spark=False,
                detect_faces=YOLO_AVAILABLE
            )
            
            start_time = time.time()
            
            results = scanner.scan_directory(
                directory_path=str(data_dir),
                recursive=True,
                sample_size=None
            )
            
            elapsed_time = time.time() - start_time
            
            if not results.empty:
                reports_dir = data_dir / "reports"
                reports_dir.mkdir(parents=True, exist_ok=True)
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                base_path = reports_dir / f"pdn_report_{timestamp}"
                
                scanner.save_report_csv(results, str(base_path.with_suffix('.csv')))
                scanner.save_report_json(results, str(base_path.with_suffix('.json')))
                scanner.save_report_markdown(results, str(base_path.with_suffix('.md')))
                
                stats = scanner.get_statistics(results)
                
                print("\n" + "="*60)
                print("СТАТИСТИКА")
                print("="*60)
                print(f"Всего файлов: {stats['total_files']}")
                print(f"Файлов с ПДн: {stats['files_with_pdn']}")
                print(f"Всего находок: {stats['total_pdn_hits']}")
                print(f"Время выполнения: {elapsed_time:.2f} сек")
                print(f"\nОтчеты сохранены в: {reports_dir}")
                
                # Показываем таблицу с 5 колонками
                print("\n" + "="*60)
                print("ФАЙЛЫ С ПЕРСОНАЛЬНЫМИ ДАННЫМИ (5 колонок)")
                print("="*60)
                
                columns = ['путь', 'категории_ПДн', 'количество_находок', 'УЗ', 'формат_файла']
                display_df = results[columns].copy()
                display_df['путь'] = display_df['путь'].apply(lambda x: Path(x).name)
                
                print(display_df.head(20).to_string(index=False))
            else:
                print("Нет результатов")
                
        except Exception as e:
            print(f"Ошибка: {e}")
            import traceback
            traceback.print_exc()
    else:
        # Режим командной строки
        def main_cli():
            parser = argparse.ArgumentParser(description="Сканер ПДн с поддержкой Spark и YOLO")
            parser.add_argument("--input", "-i", help="Директория для сканирования")
            parser.add_argument("--output", "-o", help="Базовое имя файла отчета")
            parser.add_argument("--sample", "-s", type=int, help="Ограничение количества файлов")
            parser.add_argument("--no-recursive", action="store_true", help="Отключить рекурсивный обход")
            parser.add_argument("--workers", "-w", type=int, default=60, help="Количество потоков")
            parser.add_argument("--format", "-f", choices=['csv', 'json', 'markdown', 'all'], 
                               default='all', help="Формат отчета")
            parser.add_argument("--no-spark", action="store_true", help="Отключить использование Spark")
            parser.add_argument("--no-faces", action="store_true", help="Отключить детекцию лиц")
            
            args = parser.parse_args()
            
            if args.input is None:
                args.input = str(Path.cwd() / "data")
            
            if args.output is None:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                args.output = f"pdn_report_{timestamp}"
            
            formats = ['csv', 'json', 'markdown'] if args.format == 'all' else [args.format]
            
            scanner = PDNScanner(
                max_workers=args.workers,
                use_spark=not args.no_spark,
                detect_faces=not args.no_faces
            )
            
            start_time = time.time()
            
            results = scanner.scan_directory(
                directory_path=args.input,
                recursive=not args.no_recursive,
                sample_size=args.sample
            )
            
            elapsed_time = time.time() - start_time
            
            reports_dir = Path(args.input) / "reports"
            reports_dir.mkdir(parents=True, exist_ok=True)
            
            base_path = reports_dir / args.output
            
            if 'csv' in formats:
                scanner.save_report_csv(results, str(base_path.with_suffix('.csv')))
            if 'json' in formats:
                scanner.save_report_json(results, str(base_path.with_suffix('.json')))
            if 'markdown' in formats:
                scanner.save_report_markdown(results, str(base_path.with_suffix('.md')))
            
            stats = scanner.get_statistics(results)
            
            print(f"\nВсего файлов: {stats['total_files']}")
            print(f"Файлов с ПДн: {stats['files_with_pdn']}")
            print(f"Всего находок: {stats['total_pdn_hits']}")
            print(f"Время выполнения: {elapsed_time:.2f} сек")
            print(f"\nОтчеты сохранены в: {reports_dir}")
        
        main_cli()

СКАНЕР ПЕРСОНАЛЬНЫХ ДАННЫХ
GPU доступен: True
GPU: CUDA (NVIDIA GeForce RTX 4060 Laptop GPU)
Инициализация YOLO для детекции лиц...
Загрузка YOLO модели на устройство: cuda
✅ Загружена модель: yolov8n.pt
✅ YOLO модель загружена (GPU: True)
Сбор файлов из: c:\Users\rulan\Downloads\SamsungHacaton\data
Найдено файлов: 3306
Анализ файлов (потоков: 100)...


unknown widths : 
[0, IndirectObject(313, 0, 2205122210192)]
unknown widths : 
[0, IndirectObject(308, 0, 2205122210192)]
unknown widths : 
[0, IndirectObject(303, 0, 2205122210192)]
unknown widths : 
[0, IndirectObject(298, 0, 2205122210192)]


  Обработано 50/3306 файлов
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
  Обработано 100/3306 файлов
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB

Illegal character in Name Object (b'/FPHOYY+Times New Roman \xcf\xee\xeb\xf3\xe6\xe8\xf0\xed\xfb\xe9')
Illegal character in Name Object (b'/FPHOYY+Times New Roman \xcf\xee\xeb\xf3\xe6\xe8\xf0\xed\xfb\xe9')
unknown widths : 
[0, IndirectObject(3487, 0, 2205097026048)]
unknown widths : 
[0, IndirectObject(3482, 0, 2205097026048)]
unknown widths : 
[0, IndirectObject(3477, 0, 2205097026048)]
unknown widths : 
[0, IndirectObject(3467, 0, 2205097026048)]
unknown widths : 
[0, IndirectObject(3457, 0, 2205097026048)]
unknown widths : 
[0, IndirectObject(3452, 0, 2205097026048)]
unknown widths : 
[0, IndirectObject(3447, 0, 2205097026048)]


Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
Ultralytics 8.4.38  Python-3.13.3 torch-2.9.1+cu126 CUD

...

  Обработано 2900/3306 файлов
  Обработано 2950/3306 файлов


unknown widths : 
[0, IndirectObject(108, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(103, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(98, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(93, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(88, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(83, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(78, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(73, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(68, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(63, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(58, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(53, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(48, 0, 2205096856224)]
unknown widths : 
[0, IndirectObject(43, 0, 2205096856224)]


  Обработано 3000/3306 файлов
  Обработано 3050/3306 файлов


unknown widths : 
[0, IndirectObject(387, 0, 2205231915344)]
unknown widths : 
[0, IndirectObject(382, 0, 2205231915344)]
unknown widths : 
[0, IndirectObject(377, 0, 2205231915344)]
unknown widths : 
[0, IndirectObject(372, 0, 2205231915344)]
unknown widths : 
[0, IndirectObject(367, 0, 2205231915344)]
unknown widths : 
[0, IndirectObject(362, 0, 2205231915344)]
unknown widths : 
[0, IndirectObject(357, 0, 2205231915344)]
unknown widths : 
[0, IndirectObject(367, 0, 2205231913936)]
unknown widths : 
[0, IndirectObject(362, 0, 2205231913936)]
unknown widths : 
[0, IndirectObject(357, 0, 2205231913936)]
unknown widths : 
[0, IndirectObject(352, 0, 2205231913936)]
unknown widths : 
[0, IndirectObject(347, 0, 2205231913936)]
unknown widths : 
[0, IndirectObject(342, 0, 2205231913936)]


  Обработано 3100/3306 файлов
  Обработано 3150/3306 файлов
  Обработано 3200/3306 файлов
  Обработано 3250/3306 файлов
  Обработано 3300/3306 файлов
Анализ завершен. Обработано 3306 файлов

Пример результатов:
                                                                                     путь                                 категории_ПДн  количество_находок    УЗ формат_файла
0  c:\Users\rulan\Downloads\SamsungHacaton\data\share\Прочее\DGTU-conference-programm.rtf                                     обычные:1                   1  УЗ-4          rtf
1         c:\Users\rulan\Downloads\SamsungHacaton\data\share\Выгрузки\Сайты\page_203.html  обычные:10, государственные:2, специальные:1                  13  УЗ-1         html
2         c:\Users\rulan\Downloads\SamsungHacaton\data\share\Выгрузки\Сайты\page_405.html  обычные:47, государственные:3, специальные:1                  51  УЗ-1         html
3         c:\Users\rulan\Downloads\SamsungHacaton\data\share\Выгрузки\Сайты\page_281.html